### Transliteration Using LLMs/Indicate

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

# First, count rows (optional but useful)
chunks = pd.read_csv('drive/MyDrive/punjab_all_clean+t13n.csv.gz',
                     chunksize=1000_000,
                     low_memory = False,
                     usecols=['elector_name', 'father_or_husband_name', 'polling_station_name', 'parl_constituency', 'ac_name', 'polling_station_address', 'main_town', 'mandal', 'revenue_division', 'police_station', 'district'])
df = pd.concat([chunk for chunk in tqdm(chunks, desc="Loading")])

In [ ]:
df.head()

In [ ]:
import re

gurmukhi_pattern = re.compile(r'^[\u0A00-\u0A7F]+$')
all_gurmukhi_words = set()

for col in df.columns:
    # Get unique values for this column
    unique_vals = df[col].dropna().unique()

    # Tokenize and filter
    for val in unique_vals:
        tokens = re.split(r'[\s,.\-()]+', str(val))
        for token in tokens:
            if gurmukhi_pattern.match(token):
                all_gurmukhi_words.add(token)

    print(f"{col}: {len(all_gurmukhi_words):,} cumulative unique words")

words_df = pd.DataFrame({'word': list(all_gurmukhi_words)})
print(f"\n{len(words_df):,} total unique Gurmukhi words")

In [ ]:
words_df.to_parquet('drive/MyDrive/punjabi_words.parquet')

In [ ]:
words_df

In [ ]:
import os
%pip install indicate
import indicate

from tqdm.auto import tqdm

#!pip install --force-reinstall --no-cache-dir indicate
#!pip list

In [ ]:
os.environ["OPENAI_API_KEY"] = "API KEY HERE"

def transliterate_dataframe(
      df,
      source_column,
      output_path,
      target_column="transliterated",
      source_lang="hindi",
      target_lang="english",
      batch_size=100,
      save_every=5,
      delay=0.1
  ):
      """
      Transliterate a DataFrame column with incremental parquet saves.
      Resumes from existing progress if output_path exists.
      """
      output_path = Path(output_path)

      # Resume from existing file if present
      if output_path.exists():
          print(f"Resuming from {output_path}")
          df = pd.read_parquet(output_path)
      elif target_column not in df.columns:
          df[target_column] = None

      # Initialize transliterator
      transliterator = IndicLLMTransliterator(
          source_lang=source_lang,
          target_lang=target_lang,
          model='gpt-4o',
          provider='openai'
      )

      # Find rows that still need processing
      mask = (
          df[source_column].notna() &
          (df[source_column].str.strip() != "") &
          (df[target_column].isna() | (df[target_column] == ""))
      )
      texts = df.loc[mask, source_column].tolist()
      indices = df.loc[mask].index.tolist()

      print(f"Processing {len(texts)} remaining texts in batches of {batch_size}...")

      # Process in batches
      batches_since_save = 0
      for i in tqdm(range(0, len(texts), batch_size)):
          batch_texts = texts[i:i + batch_size]
          batch_indices = indices[i:i + batch_size]

          try:
              batch_results = transliterator.transliterate_batch(
                  batch_texts,
                  batch_size=len(batch_texts)
              )
              for idx, result in zip(batch_indices, batch_results):
                  df.at[idx, target_column] = result

          except Exception as e:
              print(f"Batch failed, falling back to individual: {e}")
              for text, idx in zip(batch_texts, batch_indices):
                  try:
                      result = transliterator.transliterate(text)
                      df.at[idx, target_column] = result
                  except Exception as e2:
                      df.at[idx, target_column] = f"[ERROR: {e2}]"

          batches_since_save += 1
          if batches_since_save >= save_every:
              df.to_parquet(output_path)
              batches_since_save = 0

          time.sleep(delay)

      # Final save
      df.to_parquet(output_path)
      print(f"Saved to {output_path}")

      return df


In [ ]:
import pandas as pd
from pathlib import Path
import time
from indicate import IndicLLMTransliterator

words_df = pd.read_parquet('drive/MyDrive/punjabi_words.parquet')

result_df = transliterate_dataframe(
    words_df,
    output_path='drive/MyDrive/punjab_transliterated.parquet',
    source_column='word',
    target_column='english_words',
    source_lang = 'punjabi',
    batch_size=200,  # Optimal for large datasets
    delay=0.05       # Fast processing
    )

In [ ]:
result_df[:200]

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

# First, count rows (optional but useful)
chunks = pd.read_csv('drive/MyDrive/punjab_all_clean+t13n.csv.gz',
                     chunksize=1000_000,
                     low_memory = False,
                     usecols=['elector_name', 'father_or_husband_name', 'polling_station_name', 'parl_constituency', 'ac_name', 'polling_station_address', 'main_town', 'mandal', 'revenue_division', 'police_station', 'district'])
df = pd.concat([chunk for chunk in tqdm(chunks, desc="Loading")])

def transliterate_text(text):
    """Replace each Gurmukhi word with its transliteration."""
    if pd.isna(text):
        return text

    text = str(text)

    # Replace each token, preserving non-Gurmukhi text
    def replace_token(match):
        token = match.group(0)
        return word_map.get(token, token)

    # Match Gurmukhi words and replace
    return re.sub(r'[\u0A00-\u0A7F]+', replace_token, text)

# Apply to each column
cols_to_transliterate = ['elector_name', 'father_or_husband_name', 'polling_station_name',
                         'parl_constituency', 'ac_name', 'polling_station_address',
                         'main_town', 'mandal', 'revenue_division', 'police_station', 'district']

In [ ]:
import re
word_map = dict(zip(result_df['word'], result_df['english_words']))

# Faster version using map on unique values first
for col in tqdm(cols_to_transliterate, desc="Transliterating columns"):
    # Get unique values
    unique_vals = df[col].dropna().unique()

    # Transliterate unique values only
    val_map = {val: transliterate_text(val) for val in unique_vals}

    # Map back
    df[f'{col}_transliterated'] = df[col].map(val_map)

In [ ]:
df.head()

In [ ]:
df.to_parquet('drive/MyDrive/punjab_transliteration_subset.parquet')